# Caso 3 · Cox07 | Tarifa completa y presupuesto
Versión docente del 25 de noviembre de 2026. [Material](https://asunmayoral.github.io/GLM/caso3/caso3_continuas_positivas.html) · [Scripts oficiales](https://github.com/asunmayoral/GLM/tree/2623c88e6633507967a355cd40a8dd403ee90b0f/caso3/scripts).

El banco de averías es el ejemplo de teoría; el encargo usa la cartera de automóvil del caso 2. Cinco preguntas, un artículo. Sexo queda fuera de la tarifa; bonus-malus solo entra con orden temporal válido. Los resultados requieren ejecución.

## Scripts de referencia
Los siete scripts oficiales se han consultado. La adaptación siguiente reproduce sus métodos sobre Cox07, no ejecuta el banco industrial. Los originales se pueden descargar en una carpeta temporal mediante la celda opcional siguiente; no se mezclan objetos ni datos de ambos estudios.

In [ ]:
#@title Descargar scripts oficiales (opcional)
ref_docente <- "2623c88e6633507967a355cd40a8dd403ee90b0f"
scripts_docentes <- c("caso3_continuas_positivas.R",paste0("unidad_3_",1:6,".R"))
ruta_docente <- file.path(tempdir(),"GLM_caso3_docente")
dir.create(ruta_docente,showWarnings=FALSE)
for (nombre in scripts_docentes) {
  download.file(paste0("https://raw.githubusercontent.com/asunmayoral/GLM/",ref_docente,"/caso3/scripts/",nombre),file.path(ruta_docente,nombre),mode="wb",quiet=TRUE)
}
list.files(ruta_docente)

In [ ]:
SEMILLA_CURSO <- 20252026L
set.seed(SEMILLA_CURSO)
options(repr.plot.width=11, repr.plot.height=6)
paquetes <- c("dplyr","ggplot2","readr","glmmTMB","DHARMa","MASS","car","statmod")
faltan <- setdiff(paquetes, rownames(installed.packages()))
if(length(faltan)) install.packages(faltan, repos="https://cloud.r-project.org")
stopifnot(all(vapply(paquetes, requireNamespace, logical(1), quietly=TRUE)))
library(dplyr)
library(ggplot2)
theme_set(theme_minimal(base_size=14))
base_equipo <- "https://raw.githubusercontent.com/asunmayoral/GLM/b05aeaecde9dca0049ea0ca55fbe162a9b964cbc/caso2/datos_equipos_auto/equipo_07/"
huellas <- c("cartera.csv"="6a8a1cd5a22ec0813be10b4d2173b28f","panel.csv"="813f20b73d15d78bb5777a21268d07c8","siniestros.csv"="9523e2bd2665c492d8dc1fbe5502d7de","cartera.rds"="949486c1f1c5b51ef43f441748e1932d")
archivo_equipo <- function(nombre) {
  ruta <- file.path("datos", nombre)
  if(!file.exists(ruta)) {
    ruta <- file.path(tempdir(), nombre)
    if(!file.exists(ruta)) download.file(paste0(base_equipo,nombre),ruta,mode="wb",quiet=TRUE)
  }
  stopifnot(unname(tools::md5sum(ruta)) == huellas[[nombre]])
  ruta
}
cartera_original <- readr::read_csv(archivo_equipo("cartera.csv"),show_col_types=FALSE)
panel <- readr::read_csv(archivo_equipo("panel.csv"),show_col_types=FALSE)
siniestros <- readr::read_csv(archivo_equipo("siniestros.csv"),show_col_types=FALSE)
cartera_rds <- readRDS(archivo_equipo("cartera.rds"))
stopifnot(!anyDuplicated(cartera_original$id_poliza),sum(cartera_original$n_siniestros)==nrow(siniestros),sum(panel$n_partes_anual)==nrow(siniestros))
set.seed(SEMILLA_CURSO)
ids_ajuste <- sample(cartera_original$id_poliza,floor(.75*nrow(cartera_original)))
ids_reserva <- setdiff(cartera_original$id_poliza,ids_ajuste)
str(cartera_rds,max.level=1)

## 0 · Construir las unidades de análisis
La exposición se mide en años con expo_anual. El coste de un parte está en euros. Los predictores son características conocidas al inicio; excluimos coste, conteos futuros y bonus-malus actual. Los costes cero se conservan en el panel, no en Gamma. La reserva se separa por póliza para evitar que un mismo cliente aparezca en ajuste y validación.

In [ ]:
rasgos <- c("id_poliza","id_agencia","id_region","edad_conductor","antiguedad_vehiculo","tipo_vehiculo","zona_circulacion","uso","km_anuales","sexo")
clientes <- cartera_original |> select(all_of(rasgos)) |> mutate(across(c(id_agencia,id_region,tipo_vehiculo,zona_circulacion,uso,sexo),factor), edad10=edad_conductor/10, veh10=antiguedad_vehiculo/10, km10=km_anuales/10000)
stopifnot(all(is.finite(siniestros$coste_k)),all(siniestros$coste_k>=0),!anyDuplicated(panel[c("id_poliza","anio")]))
partes <- left_join(siniestros,clientes,by="id_poliza")
anual <- panel |> left_join(siniestros |> group_by(id_poliza,anio) |> summarise(coste=sum(coste_k),.groups="drop"),by=c("id_poliza","anio")) |> mutate(coste=coalesce(coste,0)) |> left_join(clientes,by="id_poliza")
stopifnot(abs(sum(anual$coste)-sum(partes$coste_k))<.01,all(anual$expo_anual>=0),all(anual$coste[anual$expo_anual==0]==0))
vars <- c("edad10","veh10","tipo_vehiculo","zona_circulacion","uso","km10")
stopifnot(all(complete.cases(clientes[vars])))
positivos <- filter(partes,coste_k>0)
print(c(partes=nrow(partes),ceros=sum(partes$coste_k==0),polizas=nrow(clientes),anualidades=nrow(anual)))
A <- filter(positivos,id_poliza %in% ids_ajuste)
V <- filter(positivos,id_poliza %in% ids_reserva)
P <- filter(anual,expo_anual>0,id_poliza %in% ids_ajuste)
Q <- filter(anual,expo_anual>0,id_poliza %in% ids_reserva)
f <- reformulate(vars,"coste_k")
g <- reformulate(vars,"coste")
ggplot(positivos,aes(coste_k))+geom_histogram(bins=50,fill="gold",colour="white")+labs(x="Coste por parte (€)",y="Partes")
ggplot(positivos,aes(tipo_vehiculo,coste_k))+geom_violin(fill="gold",alpha=.6)+geom_boxplot(width=.12)+scale_y_log10()+labs(y="Coste (€), escala log")

## 3.1 · Gamma: severidad
🔰 E[Y|x] = μ; Var(Y|x) = φ μ². Con enlace log, μ = exp(η), η = Xβ. Un incremento unitario de un predictor multiplica la media por exp(β). Edad10 representa diez años. El enlace inverso es otra parametrización y puede producir predictores no admisibles. Comparamos con gaussiano y Gamma inverso, sin confundir significación con capacidad predictiva.

In [ ]:
m0 <- glm(f,data=A,family=gaussian())
mG <- glm(f,data=A,family=Gamma("log"))
mI <- try(glm(f,data=A,family=Gamma("inverse"),start=c(1/mean(A$coste_k),rep(0,length(coef(mG))-1))),silent=TRUE)
summary(mG)
exp(cbind(razon=coef(mG),confint.default(mG)))
drop1(mG,test="F")
AIC(m0,mG)
if(inherits(mI,"glm") && mI$converged) print(AIC(mG,mI))
diagnostico <- function(m) {
  plot(fitted(m),residuals(m,type="pearson"),xlab="Media ajustada",ylab="Residuo Pearson",pch=16,col="#77777755")
  abline(h=0,col="tomato")
}
diagnostico(mG)
pr <- predict(mG,newdata=V,type="link",se.fit=TRUE)
head(data.frame(media=exp(pr$fit),inf=exp(pr$fit-1.96*pr$se.fit),sup=exp(pr$fit+1.96*pr$se.fit)))

### P1 · Selección y diagnóstico (script 3.1)
Comparamos modelos con contraste F de tipo II. Revisamos curvatura, residuos y cola. El modelo ampliado solo se conserva si la evidencia lo justifica. Los intervalos de confianza estiman la media; los de predicción describen un parte individual.

In [ ]:
print(car::Anova(mG,type=2,test.statistic="F"))
mCurva <- update(mG,.~.+I(edad10^2)+I(veh10^2))
print(anova(mG,mCurva,test="F"))
plot(DHARMa::simulateResiduals(mG,n=250))
phiG <- summary(mG)$dispersion
perfiles <- data.frame(perfil=c("A Joven urbano","B Familiar rural","C Moto urbana","D Furgoneta comercial","E Conductor mayor"),edad10=c(23,45,30,40,70)/10,veh10=c(8,6,4,10,15)/10,km10=c(15000,12000,8000,35000,6000)/10000,tipo_vehiculo=c("turismo","turismo","moto","furgoneta","turismo"),zona_circulacion=c("urbana","rural","urbana","mixta","mixta"),uso=c("particular","particular","particular","comercial","particular"),expo_anual=1)
for(v in c("tipo_vehiculo","zona_circulacion","uso")) perfiles[[v]] <- factor(perfiles[[v]],levels=levels(clientes[[v]]))
prG <- predict(mG,perfiles,type="link",se.fit=TRUE)
print(data.frame(perfil=perfiles$perfil,media=exp(prG$fit),ICinf=exp(prG$fit-1.96*prG$se.fit),ICsup=exp(prG$fit+1.96*prG$se.fit),IPinf=qgamma(.025,shape=1/phiG,scale=exp(prG$fit)*phiG),IPsup=qgamma(.975,shape=1/phiG,scale=exp(prG$fit)*phiG)))

## 3.2 · Familia, escala
La varianza crece con μ: normal (p=0), Gamma (p=2), inversa gaussiana (p=3). El modelo lognormal ajusta log(Y), no Y. El factor de Duan corrige la retransfomación de la media bajo error logarítmico homogéneo. Para comparar AIC en euros se añade 2 Σ log(Y) al AIC lognormal. La pendiente de la nube media-varianza es descriptiva, no una prueba independiente.

In [ ]:
mIG <- glm(f,data=A,family=inverse.gaussian("log"))
mLN <- lm(update(f,log(coste_k)~.),data=A)
duan <- mean(exp(residuals(mLN)))
comparacion <- data.frame(modelo=c("Gamma","Inversa gaussiana","Lognormal"),AIC=c(AIC(mG),AIC(mIG),AIC(mLN)+2*sum(log(A$coste_k))))
print(comparacion)
preds <- list(Gamma=predict(mG,V,type="response"),IG=predict(mIG,V,type="response"),LN=exp(predict(mLN,V))*duan)
metricas <- function(y,p) c(MAE=mean(abs(y-p)),RMSE=sqrt(mean((y-p)^2)),sesgo=mean(p-y),observado=sum(y),predicho=sum(p))
print(sapply(preds,function(p) metricas(V$coste_k,p)))
nube <- A |> mutate(grupo=ntile(fitted(mG),10)) |> group_by(grupo) |> summarise(mu=mean(coste_k),var=var(coste_k),.groups="drop")
ggplot(nube,aes(mu,var))+geom_point(size=3,colour="tomato")+geom_smooth(method="lm",se=FALSE)+scale_x_log10()+scale_y_log10()


### P2 · Incertidumbre de la retransfomación y cola
El smearing de Duan estima E[exp(ε)]. Remuestreamos pólizas completas para conservar dependencia. Los intervalos bootstrap de la media incorporan la incertidumbre del factor de retransfomación. La cola se evalúa con el mismo umbral en reserva.

In [ ]:
set.seed(SEMILLA_CURSO)
grupos <- split(A,A$id_poliza)
boot_duan <- replicate(200,{
  d <- do.call(rbind,sample(grupos,length(grupos),replace=TRUE))
  fit <- lm(update(f,log(coste_k)~.),data=d)
  exp(predict(fit,perfiles))*mean(exp(residuals(fit)))
})
print(t(apply(boot_duan,1,quantile,c(.025,.975))))
umbral <- quantile(A$coste_k,.95)
print(sapply(preds,function(p) metricas(V$coste_k[V$coste_k>=umbral],p[V$coste_k>=umbral])))
plot(DHARMa::simulateResiduals(mIG,n=250))
plot(mLN,which=1:2)

## 3.3 · P3 | Los tres caminos
1. Frecuencia × severidad.
2. Probabilidad de coste positivo × coste condicionado a ser positivo.
3. Tweedie directa.

El script oficial usa offset log(exposición) en Tweedie. Lo reproducimos y contrastamos con la variante de tasa ponderada: no tienen idéntica varianza. El modelo binomial usa enlace cloglog con offset para que la probabilidad respete la duración. El modelo de coste positivo estima libremente el efecto de log(exposición). Si existen partes de importe cero, se modela la presencia de coste, no la presencia de parte. La frecuencia usada aquí reproduce una NB con los rasgos seleccionados; falta importar la tarifa final exacta del caso 2 si difiere.

In [ ]:
P$tasa <- P$coste/P$expo_anual
Q$tasa <- Q$coste/Q$expo_anual
ft <- reformulate(c(vars,"offset(log(expo_anual))"),"coste")
mT <- glmmTMB::glmmTMB(ft,data=P,family=glmmTMB::tweedie(link="log"))
mTasa <- glmmTMB::glmmTMB(reformulate(vars,"tasa"),data=P,weights=expo_anual,family=glmmTMB::tweedie(link="log"))
print(glmmTMB::family_params(mT))
fn <- reformulate(c(vars,"offset(log(expo_anual))"),"n_partes_anual")
mF <- MASS::glm.nb(fn,data=P)
mHay <- glm(reformulate(c(vars,"offset(log(expo_anual))"),"I(coste > 0)"),data=P,family=binomial("cloglog"))
mPos <- glm(reformulate(c(vars,"log(expo_anual)"),"coste"),data=filter(P,coste>0),family=Gamma("log"))
probpos <- mean(partes$coste_k[partes$id_poliza %in% ids_ajuste]>0)
tres_caminos <- function(d) data.frame(FrecSev=predict(mF,d,type="response")*predict(mG,d,type="response")*probpos,DosPartes=predict(mHay,d,type="response")*predict(mPos,d,type="response"),Tweedie=predict(mT,d,type="response"))
predQ <- tres_caminos(Q)
print(sapply(predQ,function(p) metricas(Q$coste,p)))
print(cbind(perfiles["perfil"],tres_caminos(perfiles)))
plot(DHARMa::simulateResiduals(mT,n=250))
cal <- Q |> mutate(pred=predQ$Tweedie,grupo=ntile(pred/expo_anual,10)) |> group_by(grupo) |> summarise(obs=sum(coste)/sum(expo_anual),pred=sum(pred)/sum(expo_anual),.groups="drop")
ggplot(cal,aes(pred,obs))+geom_abline(slope=1,intercept=0)+geom_point(colour="tomato",size=3)
print(metricas(Q$coste,predict(mTasa,Q,type="response")*Q$expo_anual))

## 3.4 · P4 | Bonus-malus y credibilidad
El nivel previo del último año estima la corrección sobre la ficha. Excluimos pólizas con un solo año observado por fuga de información. La referencia es nivel 3. El nivel actual solo se aplica a una renovación futura. La corrección individual del GLMM usa todo el historial observado y no debe multiplicarse automáticamente por bonus-malus: son dos alternativas que podrían duplicar penalizaciones.
La comparación predictiva de los tres caminos se hace en P3; para P4 y P5 usamos provisionalmente Tweedie, hasta que los resultados justifiquen una elección.

In [ ]:
valido <- function(m) inherits(m,"glmmTMB") && isTRUE(m$sdr$pdHess) && isTRUE(m$fit$convergence==0)
bm <- cartera_original |> select(id_poliza,bonus_malus_prev,bonus_malus_act,baja) |> mutate(across(c(bonus_malus_prev,bonus_malus_act),~factor(as.character(.x),levels=as.character(1:5))))
ultimo <- anual |> group_by(id_poliza) |> filter(anio==max(anio),n_distinct(anio)==1L) |> ungroup() |> left_join(bm,by="id_poliza")
# El año máximo debe ser mayor que uno y tener exposición positiva.
ultimo <- filter(ultimo,anio>1,expo_anual>0)
ultimo$bonus_malus_prev <- relevel(droplevels(ultimo$bonus_malus_prev),ref="3")
mBM0 <- glmmTMB::glmmTMB(ft,data=ultimo,family=glmmTMB::tweedie(link="log"))
mBM <- glmmTMB::glmmTMB(update(ft,.~.+bonus_malus_prev),data=ultimo,family=glmmTMB::tweedie(link="log"))
print(AIC(mBM0,mBM))
print(exp(glmmTMB::fixef(mBM)$cond))
print(exp(confint(mBM,parm="beta_",method="wald")))
hist <- filter(anual,expo_anual>0)
mHist <- glmmTMB::glmmTMB(update(ft,.~.+(1|id_poliza)),data=hist,family=glmmTMB::tweedie(link="log"))
print(glmmTMB::VarCorr(mHist))
if(valido(mHist)) {
  efectos <- glmmTMB::ranef(mHist)$cond$id_poliza
  cred <- data.frame(id_poliza=as.integer(rownames(efectos)),correccion=exp(efectos[,1])) |> left_join(bm,by="id_poliza") |> left_join(hist |> group_by(id_poliza) |> summarise(expo=sum(expo_anual),.groups="drop"),by="id_poliza")
  print(head(cred))
  print(ggplot(cred,aes(bonus_malus_act,correccion))+geom_violin(fill="gold")+geom_boxplot(width=.12)+geom_hline(yintercept=1))
  print(ggplot(cred,aes(expo,log(correccion)))+geom_point(alpha=.3)+geom_hline(yintercept=0))
  plot(DHARMa::simulateResiduals(mHist,n=250))
}

## 3.5 · P5 | Presupuesto y tarifa de renovación
Cartera viva: baja = 0. Año completo, edad del conductor y vehículo aumentan un año. Se usa el nivel actual como nivel previo del próximo ejercicio. Las fichas nuevas entran en nivel 3. Los parámetros se reajustan con los datos disponibles una vez cerrada la comparación P3; P4 estima bonus-malus exclusivamente sobre el último año válido.
La simulación Tweedie sigue la representación Poisson–Gamma del script docente. Separamos incertidumbre de β y variabilidad del coste. No incluimos incertidumbre de p, φ, bajas futuras ni inflación. El GLMM de historial constituye una alternativa a bonus-malus, no una corrección adicional.

In [ ]:
stopifnot(valido(mBM))
viva <- clientes |> left_join(bm,by="id_poliza") |> filter(baja==0) |> mutate(edad10=edad10+.1,veh10=veh10+.1,expo_anual=1)
niveles <- levels(ultimo$bonus_malus_prev)
viva$bonus_malus_prev <- factor(as.character(viva$bonus_malus_act),levels=niveles)
stopifnot(!anyNA(viva$bonus_malus_prev))
perfiles$bonus_malus_prev <- factor("3",levels=niveles)
print(data.frame(perfil=perfiles$perfil,prima=predict(mBM,perfiles,type="response")))
mu <- predict(mBM,viva,type="response")
b <- glmmTMB::fixef(mBM)$cond
X <- model.matrix(delete.response(terms(mBM)),viva)
X <- X[,names(b),drop=FALSE]
Vb <- vcov(mBM)$cond
grad <- colSums(X*mu)
se <- sqrt(drop(t(grad)%*%Vb%*%grad))
print(c(media=sum(mu),ICinf=max(0,sum(mu)-1.96*se),ICsup=sum(mu)+1.96*se))
p <- glmmTMB::family_params(mBM)[[1]]
phi <- sigma(mBM)
stopifnot(p>1,p<2,phi>0)
rtweedie_pg <- function(mu,p,phi) {
  N <- rpois(length(mu),mu^(2-p)/(phi*(2-p)))
  rgamma(length(mu),shape=N*(2-p)/(p-1),scale=phi*(p-1)*mu^(p-1))
}
set.seed(SEMILLA_CURSO)
B <- MASS::mvrnorm(1000,b,Vb)
azar <- replicate(1000,sum(rtweedie_pg(mu,p,phi)))
total <- apply(B,1,function(beta) sum(rtweedie_pg(exp(drop(X%*%beta)),p,phi)))
print(rbind(proceso=quantile(azar,c(.05,.5,.9,.95)),total=quantile(total,c(.05,.5,.9,.95))))
ggplot(data.frame(presupuesto=total),aes(presupuesto))+geom_histogram(bins=45,fill="gold",colour="white")+geom_vline(xintercept=quantile(total,.9),colour="tomato")
if(valido(mHist)) {
  cond <- predict(mHist,viva,type="response",allow.new.levels=TRUE)
  fija <- predict(mHist,viva,type="response",re.form=NA,allow.new.levels=TRUE)
  v <- glmmTMB::VarCorr(mHist)$cond$id_poliza[1,1]
  print(c(historial_condicional=sum(cond),efecto_cero=sum(fija),marginal_nuevos=sum(fija*exp(v/2))))
}

### Validación hacia atrás
Separamos el último año de cada póliza con historial. El modelo de credibilidad aprende únicamente de los años anteriores, y predice los costes observados del último. Esta prueba no usa niveles actuales. La duración observada se conoce retrospectivamente: no es una predicción de baja.

In [ ]:
hist <- hist |> group_by(id_poliza) |> mutate(ultimo_anio=anio==max(anio),varios=n()>1) |> ungroup()
train_h <- filter(hist,!ultimo_anio | !varios)
test_h <- filter(hist,ultimo_anio,varios)
mBack <- glmmTMB::glmmTMB(update(ft,.~.+(1|id_poliza)),data=train_h,family=glmmTMB::tweedie(link="log"))
if(valido(mBack)) print(metricas(test_h$coste,predict(mBack,test_h,type="response",allow.new.levels=TRUE)))
print(metricas(test_h$coste,predict(mBack,test_h,type="response",re.form=NA,allow.new.levels=TRUE)))

## 3.6 · Estudio de caso | Artículo de investigación
El encargo oficial contiene cinco preguntas obligatorias (P1–P5), no una elección de tema. Integrar los resultados de las cinco sesiones en un artículo: objetivo, datos, métodos, resultados, discusión y conclusiones. Explicar elección de familia, enlace, estructura aleatoria, exposición y ceros. La validación contra el generador no es requisito de esta versión. No presentar cifras antes de ejecutar y revisar los ajustes.

**Trazabilidad:** script maestro → preparación; 3.1 → Gamma, F, DHARMa y perfiles; 3.2 → familia, jacobiano y smearing bootstrap; 3.3 → tres caminos y Tweedie; 3.4 → bonus-malus y credibilidad; 3.5 → presupuesto, simulación y validación hacia atrás; 3.6 → encargo sin código analítico propio.

Pendiente: escoger el modelo definitivo con evidencia, contrastar la tarifa exacta de frecuencia del caso 2, revisar sensibilidad de dispersión y comprobar la ejecución completa.

In [ ]:
sessionInfo()